# Reproducing results on the Sage paper

## Packages

In [1]:
! pip uninstall sage -y
! pip install -e ../

Found existing installation: sage 0.0.1
Uninstalling sage-0.0.1:
  Successfully uninstalled sage-0.0.1
Obtaining file:///home/nnarenraju/Research/sage
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sage (pyproject.toml) ... done
  Created wheel for sage: filename=sage-0.0.1-0.editable-py3-none-any.whl size=3660 sha256=2f391de1eb82b1616ebe5e03c5caef7d145e95dfa3cac04762f02eb19d691b93
  Stored in directory: /tmp/pip-ephem-wheel-cache-zcdkdock/wheels/9b/47/cf/bfd99e705866b88810d9db84a85969530a2783f51228948a36
Successfully built sage


In [2]:
import h5py
import numpy as np

## Get Segments for Downloading Real Noise 

In [3]:
from sage.data.primer import TimelineQuery, get_all_detnames, get_all_runnames

In [4]:
print(get_all_detnames())
print(get_all_runnames())

{'V1', 'L1', 'H2', 'H1', 'G1', 'K1'}
['O1', 'O2', 'O3GK', 'O3a', 'O3b', 'O4a', 'O4b1Disc', 'O4b2Disc', 'O4b3Disc', 'S5', 'S6']


In [5]:
tq = TimelineQuery(detector=["H1", "L1", "V1"], 
                   observing_run=["O3a",],
                   start = 1238166018,
                   end = 1238176018,
                   auto_clean_empty_timelines=True)
                                
tq.download_segments()

2026-01-19 22:29:25 | INFO     | sage.data.download.get_segments:287 | Getting all segments between 1238166018-1238176018 for all detectors in the correct observing run


In [6]:
tq.prune_segments(
    rm_short_segments = True,
    rm_min_duration = 22.0,
    rm_allevents = True,
    rm_window_length = 30,
)

2026-01-19 22:29:27 | INFO     | sage.data.download.get_segments:526 | Removing all events (obtained from GWOSC) from segments
2026-01-19 22:29:27 | INFO     | sage.data.download.get_segments:527 | [event_gps - rm_length, event_gps + rm_length] will be removed
2026-01-19 22:29:29 | INFO     | sage.data.download.get_segments:550 | Removing segments below the min duration of 22.0


In [7]:
tq.timeline

array([('H1', 'H1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1.23816602e+09, 1.23817055e+09],
              [1.23817095e+09, 1.23817293e+09],
              [1.23817299e+09, 1.23817602e+09]]))                                                       ,
       ('L1', 'L1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1.23816602e+09, 1.23817029e+09],
              [1.23817543e+09, 1.23817602e+09]]))                                                       ,
       ('V1', 'V1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1.23816602e+09, 1.23817602e+09]]))],
      dtype=[('detector', '<U2'), ('flag', '<U10'), ('start_time', '<f8'), ('end_time', '<f8'), ('observing_run', '<U10'), ('segments', 'O')])

## Download Segments from GWOSC

In [8]:
from sage.data.primer import DataReleaseDownloader

In [ ]:
drd = DataReleaseDownloader(
    segments_metadata=tq.timeline,
    save_parent_dir="./",
    noise_low_freq_cutoff = 15.0,
    minimum_segment_duration = 22.0,
    corrupt_trim_length = 0.2,
    max_download_retries = 15,
    retry_delay = 0.5,
    num_workers = 4,
    make_monolithic_file = True,
    sample_rate = 2048.0,
)

drd.download()

In [9]:
foo = h5py.File('./data_release/data_H1_O3a.h5', 'r')
print(list(foo.keys()))

['metadata', 'segments']


In [10]:
foo['segments'].keys()

<KeysViewHDF5 ['00000', '00001', '00002']>

In [11]:
# TODO: Find out why this is slightly smaller than expected
num = np.array(foo['segments/00000']).shape[0] + np.array(foo['segments/00001']).shape[0] + np.array(foo['segments/00002']).shape[0]
print((num+0)/2048.)

9535.798828125


In [12]:
foo['segments/00000'].attrs.keys()

<KeysViewHDF5 ['detector', 'gps_end', 'gps_start', 'noise_low_freq_cutoff', 'nsamples', 'old_sample_rate', 'run', 'sample_rate', 'trim']>

## Get Noise PSD Estimates from Real Noise

In [13]:
from sage.data.noise import GenerateRealNoise

In [14]:
noise_gen = GenerateRealNoise("./data_release/data_H1_O3a.h5")

In [21]:
noise_gen(2048)

array([-1.06295498e-20,  1.16024759e-20,  2.71945373e-21, ...,
       -9.36859497e-22,  1.66786317e-20,  5.70809729e-21],
      shape=(2048,))